In [ ]:
# Install gdown for downloading files from Google Drive
!pip install gdown -q
!pip install geopy tqdm -q

In [ ]:
import pandas as pd
import gdown

# Direct file ID for the CSV file
file_id = '1LDxDVDp3Q5KkMg1Ve9IwPd1cN0kqiqiq'

# Define the output filename for the downloaded CSV
output_filename = 'puskesmas_jatim_alamat.csv'

# Download the file using gdown
gdown.download(id=file_id, output=output_filename, quiet=False)

print(f"File downloaded successfully as '{output_filename}'")

Downloading...
From: https://drive.google.com/uc?id=1LDxDVDp3Q5KkMg1Ve9IwPd1cN0kqiqiq
To: /content/puskesmas_jatim_alamat.csv
100%|██████████| 121k/121k [00:00<00:00, 62.4MB/s]

File downloaded successfully as 'puskesmas_jatim_alamat.csv'


In [ ]:
# Load the downloaded CSV file into a pandas DataFrame
df_drive = pd.read_csv(output_filename)

# Display the first 5 rows of the DataFrame
print("First 5 rows of the loaded DataFrame:")
display(df_drive.head())

First 5 rows of the loaded DataFrame:


,no,kode_puskesmas,nama_puskesmas,alamat,kecamatan,kabupaten,provinsi,kemampuan_pelayanan,karakteristik_wilayah
0,4874,35010200001,DONOROJO,Jl. Donorojo-Kalak RT 02 RW 01 Dsn. Druju Ds. ...,Donorojo,Pacitan,Jawa Timur,Rawat Inap,Perdesaan
1,4875,35010200002,KALAK,RT 04 RW 02 Dsn. Krajan Kulon Ds. Kalak Kec. D...,Donorojo,Pacitan,Jawa Timur,Non Rawat Inap,Terpencil
2,4876,35010200003,PUNUNG,Jl. Solo-Pacitan KM 17 RT 02 RW 05 Dsn. Margor...,Punung,Pacitan,Jawa Timur,Rawat Inap,Perdesaan
3,4877,35010200004,GONDOSARI,Jl. Gondosari-Ploso RT 01 RW 01 Dsn. Krajan Ds...,Punung,Pacitan,Jawa Timur,Rawat Inap,Terpencil
4,4878,35010200005,PRINGKUKU,Jl.Tumpak Rinjing KM 22 RT 01 RW 09 Dsn. Bulu ...,Pringkuku,Pacitan,Jawa Timur,Non Rawat Inap,Perdesaan


Geocode Aldi

In [ ]:
"""
Script Geocoding Alamat Puskesmas Jawa Timur -> Koordinat (Lat, Long)
=======================================================================
Cara pakai:
1. Install library: pip install geopy pandas tqdm
2. Letakkan file 'puskesmas_jatim_alamat.csv' di folder yang sama
3. Jalankan: python geocoding_puskesmas.py
4. Hasil otomatis tersimpan di 'puskesmas_jatim_koordinat.csv'

Menggunakan Nominatim (OpenStreetMap) - GRATIS, tanpa API key.
Ada rate limit 1 request/detik, jadi 973 data akan makan waktu ~20-30 menit.
"""

import pandas as pd
from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter
from tqdm import tqdm
import time

# ============ KONFIGURASI ============
INPUT_FILE = "puskesmas_jatim_alamat.csv"
OUTPUT_FILE = "puskesmas_jatim_koordinat.csv"
USER_AGENT = "riset_puskesmas_jatim_app"  # boleh diganti nama app bebas

# ============ SETUP GEOCODER ============
geolocator = Nominatim(user_agent=USER_AGENT)
# min_delay_seconds=1 wajib untuk mematuhi kebijakan penggunaan Nominatim
geocode = RateLimiter(geolocator.geocode, min_delay_seconds=1, max_retries=2, error_wait_seconds=5)

# ============ BACA DATA ============
df = pd.read_csv(INPUT_FILE)
print(f"Total data: {len(df)} puskesmas")

# ============ SIAPKAN QUERY ALAMAT ============
# Gabungkan alamat + kecamatan + kabupaten + provinsi + "Indonesia" agar geocoding lebih akurat
df["query_alamat"] = (
    df["nama_puskesmas"].fillna("") + ", "
    + df["kecamatan"].fillna("") + ", "
    + df["kabupaten"].fillna("") + ", "
    + "Jawa Timur, Indonesia"
)

# ============ PROSES GEOCODING ============
latitudes = []
longitudes = []
status = []

print("Memulai geocoding... (Estimasi waktu: ~1 detik per baris)")
for query in tqdm(df["query_alamat"], desc="Geocoding"):
    try:
        location = geocode(query)
        if location:
            latitudes.append(location.latitude)
            longitudes.append(location.longitude)
            status.append("Ditemukan")
        else:
            latitudes.append(None)
            longitudes.append(None)
            status.append("Tidak ditemukan")
    except Exception as e:
        latitudes.append(None)
        longitudes.append(None)
        status.append(f"Error: {e}")

df["latitude"] = latitudes
df["longitude"] = longitudes
df["status_geocoding"] = status

# ============ SIMPAN HASIL ============
df.to_csv(OUTPUT_FILE, index=False)

found = df["status_geocoding"].eq("Ditemukan").sum()
print(f"\nSelesai! {found} dari {len(df)} data berhasil ditemukan koordinatnya.")
print(f"Hasil disimpan di: {OUTPUT_FILE}")
print(f"\nUntuk data yang 'Tidak ditemukan', coba geocoding manual via Google Maps")
print("atau perbaiki nama kecamatan/kabupaten yang mungkin typo.")

Total data: 973 puskesmas
Memulai geocoding... (Estimasi waktu: ~1 detik per baris)


Geocoding: 100%|██████████| 973/973 [16:12<00:00,  1.00it/s]


Selesai! 565 dari 973 data berhasil ditemukan koordinatnya.
Hasil disimpan di: puskesmas_jatim_koordinat.csv

Untuk data yang 'Tidak ditemukan', coba geocoding manual via Google Maps
atau perbaiki nama kecamatan/kabupaten yang mungkin typo.


Geocode Pandi

In [ ]:
# ============================================================
# GEOCODING PUSKESMAS JAWA TIMUR - VERSI 3 (MULAI DARI AWAL)
# Untuk Google Colab
# ============================================================

# ---- 1. INSTALL LIBRARY (jalankan sekali di awal) ----
!pip install geopy -q

import pandas as pd
import re
from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter
from tqdm import tqdm

# ============ 2. KONFIGURASI ============
# File ini diambil langsung dari panel "Files" di Colab
INPUT_ALAMAT = "puskesmas_jatim_alamat.csv"
OUTPUT_FILE = "puskesmas_jatim_koordinat_v3.csv"

# GANTI dengan identitas bebas (wajib diisi sesuai kebijakan Nominatim, tidak boleh kosong/default)
USER_AGENT = "riset_ta_puskesmas_jatim_kelompok5_2026"

# ============ 3. BACA DATA ============
df = pd.read_csv(INPUT_ALAMAT)
df["latitude"] = None
df["longitude"] = None
df["status_geocoding"] = None

print(f"Total data: {len(df)} puskesmas")

# ============ 4. EKSTRAKSI NAMA DESA/KELURAHAN DARI ALAMAT ============
def extract_desa(alamat):
    """
    Mengambil nama desa/kelurahan dari teks alamat.
    Nama desa lebih mudah dikenali Nominatim dibanding nama puskesmas,
    karena desa adalah unit batas administrasi resmi di OSM.
    Sengaja TIDAK menangkap 'Dsn.' (dusun) karena itu bukan unit administrasi desa.
    """
    if pd.isna(alamat):
        return None
    patterns = [
        r'\b(?:Ds\.?|Desa)\s+([A-Za-z0-9\.\'\-\s]+?)(?=,|\s+RT|\s+RW|\s+Kec\.|\s+Kecamatan|$)',
        r'\b(?:Kel\.?|Kelurahan)\s+([A-Za-z0-9\.\'\-\s]+?)(?=,|\s+RT|\s+RW|\s+Kec\.|\s+Kecamatan|$)',
    ]
    for p in patterns:
        m = re.search(p, alamat, re.IGNORECASE)
        if m:
            hasil = m.group(1).strip(" .,-")
            if hasil:
                return hasil
    return None

df["desa_extracted"] = df["alamat"].apply(extract_desa)
print(f"Berhasil ekstrak nama desa/kelurahan: {df['desa_extracted'].notna().sum()} baris")

# ============ 5. SETUP GEOCODER ============
geolocator = Nominatim(user_agent=USER_AGENT)
# min_delay_seconds=1 WAJIB sesuai kebijakan penggunaan Nominatim (max 1 request/detik)
geocode = RateLimiter(geolocator.geocode, min_delay_seconds=1, max_retries=2, error_wait_seconds=5)

def coba_geocode(query):
    try:
        location = geocode(query, country_codes="id", exactly_one=True)
        if location:
            return location.latitude, location.longitude
    except Exception:
        pass
    return None, None

# ============ 6. PROSES GEOCODING BERTINGKAT (FALLBACK) ============
# Tahap A : "Puskesmas {nama_puskesmas}" + kecamatan + kabupaten -> mencari POI persis
# Tahap A2: "Puskesmas {nama_puskesmas} {kabupaten}" (tanpa kecamatan) -> penanganan
#           kasus nama puskesmas yang sama tapi beda kabupaten, misal
#           "Puskesmas Candi" ada di Pacitan DAN di Sidoarjo. Query dengan kecamatan
#           kadang gagal karena nama kecamatan versi OSM sedikit berbeda, jadi query
#           "Puskesmas Candi Sidoarjo" dicoba sebagai alternatif yang lebih umum dipakai
# Tahap B : Desa/Kelurahan + Kecamatan + Kabupaten               -> presisi tinggi
# Tahap C : Kecamatan + Kabupaten                                 -> perkiraan pusat kecamatan
# Tahap D : Kabupaten/Kota saja                                   -> perkiraan kasar (fallback terakhir)

print(f"\nMemulai geocoding bertingkat untuk {len(df)} baris...")
print("Estimasi waktu: ~1-5 detik per baris (tergantung berapa tahap fallback dicoba)\n")

for idx in tqdm(df.index, desc="Geocoding"):
    row = df.loc[idx]
    nama = row["nama_puskesmas"]
    kec = row["kecamatan"]
    kab = row["kabupaten"]
    desa = row["desa_extracted"]

    lat, lon, tahap = None, None, None

    # Tahap A: nama puskesmas + kecamatan + kabupaten
    if pd.notna(nama):
        query = f"Puskesmas {nama}, {kec}, {kab}, Jawa Timur, Indonesia"
        lat, lon = coba_geocode(query)
        if lat is not None:
            tahap = "Ditemukan (nama puskesmas + kecamatan)"

    # Tahap A2: nama puskesmas + kabupaten saja (tanpa kecamatan)
    # penting untuk kasus nama puskesmas kembar di kabupaten berbeda
    if lat is None and pd.notna(nama):
        query = f"Puskesmas {nama} {kab}, Jawa Timur, Indonesia"
        lat, lon = coba_geocode(query)
        if lat is not None:
            tahap = "Ditemukan (nama puskesmas + kabupaten)"

    # Tahap B: desa/kelurahan
    if lat is None and pd.notna(desa):
        query = f"{desa}, {kec}, {kab}, Jawa Timur, Indonesia"
        lat, lon = coba_geocode(query)
        if lat is not None:
            tahap = "Ditemukan (level desa)"

    # Tahap C: kecamatan
    if lat is None:
        query = f"Kecamatan {kec}, {kab}, Jawa Timur, Indonesia"
        lat, lon = coba_geocode(query)
        if lat is not None:
            tahap = "Ditemukan (level kecamatan - perkiraan)"

    # Tahap D: kabupaten
    if lat is None:
        query = f"{kab}, Jawa Timur, Indonesia"
        lat, lon = coba_geocode(query)
        if lat is not None:
            tahap = "Ditemukan (level kabupaten - perkiraan kasar)"

    if lat is not None:
        df.at[idx, "latitude"] = lat
        df.at[idx, "longitude"] = lon
        df.at[idx, "status_geocoding"] = tahap
    else:
        df.at[idx, "status_geocoding"] = "Tidak ditemukan"

# ============ 7. RINGKASAN HASIL ============
print("\n=== RINGKASAN HASIL GEOCODING ===")
print(df["status_geocoding"].value_counts())

total_ketemu = df["latitude"].notna().sum()
print(f"\nTotal berhasil : {total_ketemu} dari {len(df)} ({total_ketemu/len(df)*100:.1f}%)")

# ============ 8. SIMPAN & DOWNLOAD HASIL ============
df.to_csv(OUTPUT_FILE, index=False)
print(f"\nHasil disimpan ke: {OUTPUT_FILE}")

# File otomatis muncul di panel "Files" Colab (tersimpan di sesi ini).
# Baris di bawah ini opsional, kalau ingin langsung diunduh ke komputer:
from google.colab import files
files.download(OUTPUT_FILE)

Total data: 973 puskesmas
Berhasil ekstrak nama desa/kelurahan: 403 baris

Memulai geocoding bertingkat untuk 973 baris...
Estimasi waktu: ~1-5 detik per baris (tergantung berapa tahap fallback dicoba)



Geocoding: 100%|██████████| 973/973 [48:24<00:00,  2.98s/it]


=== RINGKASAN HASIL GEOCODING ===
status_geocoding
Ditemukan (level kabupaten - perkiraan kasar)    396
Ditemukan (nama puskesmas + kecamatan)           264
Ditemukan (level kecamatan - perkiraan)          149
Ditemukan (level desa)                            90
Ditemukan (nama puskesmas + kabupaten)            74
Name: count, dtype: int64

Total berhasil : 973 dari 973 (100.0%)

Hasil disimpan ke: puskesmas_jatim_koordinat_v3.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>